In [1]:
%pip install -q faiss-gpu

%pip install -q -U sentence_transformers

!nvidia-smi

Mon Oct 26 16:53:55 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA A100-SXM4-80GB          Off |   00000000:00:05.0 Off |                    0 |
| N/A   32C    P0             53W /  400W |       0MiB /  81920MiB |      0%      Default |
|                                         |                        |             Disabled |
+-----------------------------------------+-----

In [1]:
import os

os.environ["HF_TOKEN"] = input()

In [2]:
from datasets import load_dataset

dataset = load_dataset("pameydorke/arcanumchat-cross-platform-with-vague-queries-qwen-36-a3b-fp8")

dataset["train"]

Dataset({
    features: ['doc_id', 'link', 'title', 'text', 'query'],
    num_rows: 4738
})

In [3]:
from sentence_transformers import SentenceTransformer
from datasets import Dataset

retriever = SentenceTransformer("pameydorke/arcanumchat-retriever-bge-m3-v2")

train_split = dataset["train"]
val_split = dataset["validation"]

corpus_texts = list(dict.fromkeys(train_split["text"])) + list(dict.fromkeys(val_split["text"]))
# corpus_dataset = Dataset.from_dict({"text": corpus_texts})

# corpus_embeddings = retriever.encode(
#     corpus_texts,
#     batch_size=32,
#     show_progress_bar=True,
#     convert_to_tensor=True
# )

modules.json:   0%|          | 0.00/413 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/284 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/42.7k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/241 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/745 [00:00<?, ?B/s]

...

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/409 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/90.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/97.0 [00:00<?, ?B/s]

In [6]:
from sentence_transformers.util import mine_hard_negatives
from datasets import Dataset

train_dataset = dataset["train"].select_columns(["query", "text"])
train_dataset = train_dataset.rename_columns({
    "query": "anchor",
    "text": "positive"
})

hard_train_dataset = mine_hard_negatives(
    dataset=train_dataset,
    model=retriever,
    corpus=corpus_texts,
    num_negatives=4,

    range_min=1,
    range_max=100,
    relative_margin=0.05,
    sampling_strategy="top",
    batch_size=128,
    
    use_faiss=True,
    
    output_format="labeled-pair",
)

print(f"Original dataset size: {len(train_dataset)}")
print(f"Enriched dataset size: {len(hard_train_dataset)}")
print(hard_train_dataset[0])

Found 4725 unique queries out of 4738 total queries.
Found an average of 1.003 positives per query.


Batches:   0%|          | 0/9 [00:00<?, ?it/s]

Batches:   0%|          | 0/37 [00:00<?, ?it/s]

Querying FAISS index: 100%|██████████| 1/1 [00:00<00:00,  9.75it/s]


Negative candidates mined, preparing dataset...
Metric       Positive       Negative     Difference
Count           4,738         18,738               
Mean           0.6711         0.4402         0.2375
Median         0.6862         0.4372         0.2378
Std            0.1287         0.0903         0.1073
Min           -0.0601         0.1272         0.0081
25%            0.6181         0.3776         0.1587
50%            0.6862         0.4372         0.2378
75%            0.7537         0.4995         0.3127
Max            0.9537         0.8066         0.6354
Skipped 12,256 potential negatives (2.52%) due to the relative_margin of 0.05.
Could not find enough negatives for 210 samples (0.89%). Consider adjusting the range_max, range_min and relative_margin parameters if you'd like to find more valid negatives.
Original dataset size: 4738
Enriched dataset size: 23428
{'anchor': 'turn target into liquid', 'positive': 'Body of Water. Body of Water. Body of Water is the fourth spell in th

In [7]:
val_dataset = dataset["validation"].select_columns(["query", "text"])
val_dataset = val_dataset.rename_columns({
    "query": "anchor",
    "text": "positive"
})

hard_val_dataset = mine_hard_negatives(
    dataset=val_dataset,
    model=retriever,
    corpus=corpus_texts,
    num_negatives=4,

    range_min=1,
    range_max=100,
    relative_margin=0.05,
    sampling_strategy="top",
    batch_size=128,
    
    use_faiss=True,
    
    output_format="labeled-pair",
)

print(f"Original dataset size: {len(train_dataset)}")
print(f"Enriched dataset size: {len(hard_train_dataset)}")
print(hard_train_dataset[0])

Batches:   0%|          | 0/9 [00:00<?, ?it/s]

Batches:   0%|          | 0/5 [00:00<?, ?it/s]

Querying FAISS index: 100%|██████████| 1/1 [00:00<00:00, 599.44it/s]


Negative candidates mined, preparing dataset...
Metric       Positive       Negative     Difference
Count             539          2,129               
Mean           0.5831         0.4084         0.1811
Median         0.6124         0.4079         0.1695
Std            0.1783         0.0979         0.1220
Min           -0.0780         0.1204         0.0066
25%            0.4650         0.3458         0.0724
50%            0.6124         0.4079         0.1695
75%            0.7218         0.4710         0.2650
Max            0.9275         0.7572         0.5606
Skipped 3,299 potential negatives (6.06%) due to the relative_margin of 0.05.
Could not find enough negatives for 27 samples (1.00%). Consider adjusting the range_max, range_min and relative_margin parameters if you'd like to find more valid negatives.
Original dataset size: 4738
Enriched dataset size: 23428
{'anchor': 'turn target into liquid', 'positive': 'Body of Water. Body of Water. Body of Water is the fourth spell in the 

In [8]:
from collections import defaultdict

grouped = defaultdict(lambda: {"positive": [], "negative": []})

for anchor, document, label in zip(
    hard_val_dataset["anchor"], hard_val_dataset["positive"], hard_val_dataset["label"]
):
    key = "positive" if int(label) == 1 else "negative"
    grouped[anchor][key].append(document)

# Only keep queries that have at least one positive AND one negative
hard_eval_samples = [
    {"query": q, **docs}
    for q, docs in grouped.items()
    if docs["positive"] and docs["negative"]
]

print(f"Evaluation queries: {len(hard_eval_samples)}")

Evaluation queries: 534


In [9]:
from sentence_transformers.cross_encoder.evaluation import CrossEncoderRerankingEvaluator

evaluator = CrossEncoderRerankingEvaluator(
    samples=hard_eval_samples,
    at_k=10,
    name="val-hard-negatives",
    batch_size=64,
    show_progress_bar=True,
    always_rerank_positives=True,
)

In [10]:
from sentence_transformers import CrossEncoder

model = CrossEncoder("BAAI/bge-reranker-v2-m3", num_labels=1)

Loading weights:   0%|          | 0/393 [00:00<?, ?it/s]

In [11]:
from sentence_transformers.cross_encoder.losses import BinaryCrossEntropyLoss
from datasets import Dataset

train_dataset_formatted = Dataset.from_dict({
    "query": [p["anchor"] for p in hard_train_dataset],
    "document": [p["positive"] for p in hard_train_dataset],
    "label": [p["label"] for p in hard_train_dataset]
})

val_dataset_formatted = Dataset.from_dict({
    "query": [p["anchor"] for p in hard_val_dataset],
    "document": [p["positive"] for p in hard_val_dataset],
    "label": [p["label"] for p in hard_val_dataset]
})

train_loss = BinaryCrossEntropyLoss(model)

In [12]:
from sentence_transformers.cross_encoder import CrossEncoderTrainer, CrossEncoderTrainingArguments
from transformers import EarlyStoppingCallback

args = CrossEncoderTrainingArguments(
    warmup_steps=0.1,
    lr_scheduler_type="linear",

    num_train_epochs=1,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    gradient_accumulation_steps = 2,

    logging_steps=100,

    eval_strategy="steps",
    eval_steps=100,
    save_strategy="steps",
    save_steps=100,
    save_total_limit=2,

    load_best_model_at_end=True,
    metric_for_best_model="eval_val-hard-negatives_ndcg@10",
    greater_is_better=True,

    eval_on_start=True,
)

trainer = CrossEncoderTrainer(
    model=model,
    args=args,
    train_dataset=train_dataset_formatted,
    eval_dataset=val_dataset_formatted,

    evaluator=evaluator,
    loss=train_loss,
    # callbacks=[
    #         EarlyStoppingCallback(
    #                 early_stopping_patience=3,
    #                 early_stopping_threshold=0.001,
    #             )
    #     ],
)

In [13]:
trainer.train()

Step,Training Loss,Validation Loss,Val-hard-negatives Map,Val-hard-negatives Mrr@10,Val-hard-negatives Ndcg@10
0,No log,0.287987,0.952996,0.952996,0.965028
100,0.300738,0.266024,0.958489,0.958489,0.969094
200,0.283825,0.226266,0.961361,0.961361,0.971304
300,0.257191,0.210172,0.968165,0.968165,0.976412
400,0.230542,0.181358,0.974251,0.974251,0.980920
500,0.229892,0.192456,0.968789,0.968789,0.976858
600,0.213730,0.177972,0.973783,0.973783,0.980559
700,0.207788,0.174484,0.974719,0.974719,0.981250
733,0.207788,0.173630,0.975655,0.975655,0.981941


Batches:   0%|          | 0/42 [00:00<?, ?it/s]

Batches:   0%|          | 0/42 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Batches:   0%|          | 0/42 [00:00<?, ?it/s]

...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Batches:   0%|          | 0/42 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/393 [00:00<?, ?it/s]

TrainOutput(global_step=733, training_loss=0.24372593149807087, metrics={'train_runtime': 3726.8988, 'train_samples_per_second': 6.286, 'train_steps_per_second': 0.197, 'total_flos': 0.0, 'train_loss': 0.24372593149807087, 'epoch': 1.0})

In [14]:
model.push_to_hub("pameydorke/arcanumchat-reranker-bge-reranker-v2-m3-v2")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

'https://huggingface.co/pameydorke/arcanumchat-reranker-bge-reranker-v2-m3-v2/commit/a1be470e1f03b5d5e2226f5c883ee37ddebe6de9'